In [2]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [4]:
# data preprocessing
# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)


def removing_numbers(text):
    """Removing numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)


def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)


def normalize_text(df):
    """Normalize the text data"""
    try:
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f"Error during text normalization: {e}")
        raise

<>:33: SyntaxWarning: invalid escape sequence '\s'
<>:33: SyntaxWarning: invalid escape sequence '\s'
C:\Users\dhruv\AppData\Local\Temp\ipykernel_20380\3616341511.py:33: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text).strip()


In [5]:
df = normalize_text(df)
df.head()

,sentiment,content
0,empty,tiffanylue know listenin bad habit earlier sta...
1,sadness,layin n bed headache ughhhh waitin call
2,sadness,funeral ceremony gloomy friday
3,enthusiasm,want hang friend soon
4,neutral,dannycastillo want trade someone houston ticke...


In [10]:
df['sentiment'].value_counts()

sentiment
neutral       8638
worry         8459
happiness     5209
sadness       5165
love          3842
surprise      2187
fun           1776
relief        1526
hate          1323
empty          827
enthusiasm     759
boredom        179
anger          110
Name: count, dtype: int64

In [11]:
x = df['sentiment'].isin(['happiness','sadness'])
df = df[x]

In [12]:
df

,sentiment,content
1,sadness,layin n bed headache ughhhh waitin call
2,sadness,funeral ceremony gloomy friday
6,sadness,sleep im not thinking old friend want married ...
8,sadness,charviray charlene love miss
9,sadness,kelcouch sorry least friday
...,...,...
39986,happiness,going watch boy striped pj s hope cry
39987,happiness,gave bike thorough wash degrease grease it thi...
39988,happiness,amazing time last night mcfly incredible
39994,happiness,succesfully following tayla


In [13]:
x

0        False
1         True
2         True
3        False
4        False
         ...  
39995    False
39996    False
39997    False
39998     True
39999    False
Name: sentiment, Length: 40000, dtype: bool

In [16]:
df['sentiment'] = df['sentiment'].replace({'sadness':0,'happiness':1})
df.sample(10)

,sentiment,content
4882,0,luvinmesomed that s march it came sooo slow th...
25873,1,mastersavage cool saw link thanks
31115,1,josieinthecity lol good men watch flick w u si...
96,0,dont wanna work tomorrow get paid
36649,1,gooooooood morning camper happy mother s day
21094,1,souravghosh dream give rise reality said walk ...
17501,1,annadifilippo fun without pasty lt
38661,0,hot shower make everything better need sumbody...
11705,0,officially miss quark huhuhu heart hand partne...
34370,1,got done turning pre final project computer an...


In [17]:
vectorizer = CountVectorizer(max_features=1000)
X   = vectorizer.fit_transform(df['content'])
y = df['sentiment']

In [18]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state=42)



In [23]:
import dagshub
mlflow.set_tracking_uri('https://dagshub.com/DhruvMishra007/mlops-mini-project.mlflow')
dagshub.init(repo_owner='DhruvMishra007', repo_name='mlops-mini-project', mlflow=True)


mlflow.set_experiment("Logistic Regression Baseline")

Initialized MLflow to track repo "DhruvMishra007/mlops-mini-project"

Repository DhruvMishra007/mlops-mini-project initialized!

2026/01/13 19:42:59 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/e845b81210b54d42b408d64741ae36cb', creation_time=1768313581502, experiment_id='1', last_update_time=1768313581502, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}>

In [24]:
with mlflow.start_run():
    # Log preprocessing parameters
    mlflow.log_param("vectorizer","Bag of Words")
    mlflow.log_param("num_features",1000)
    mlflow.log_param("test_size",0.2)

    # Model building
    model = LogisticRegression()
    model.fit(X_train,y_train)

    # Log model parameters
    mlflow.log_param("model","LogisticRegression")

    # Model evaluation
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test,y_pred)
    precision = precision_score(y_test,y_pred)
    recall = recall_score(y_test,y_pred)
    f1 = f1_score(y_test,y_pred)

    # Log evaluation matrix
    mlflow.log_metric("accuracy",accuracy)
    mlflow.log_metric("precision",precision)
    mlflow.log_metric("recall",recall)
    mlflow.log_metric("f1_score",f1)

    # Log model
    mlflow.sklearn.log_model(model,"model")

    # Save and log the notebook
    import os
    notebook_path = "exp1_baseline_model.ipynb"
    os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
    mlflow.log_artifact(notebook_path)

    # print the result for verification
    print(f"accuracy: {accuracy}")
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")
    print(f"F1 Score: {f1}")


2026/01/13 19:43:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


accuracy: 0.776867469879518
Precision: 0.7690058479532164
Recall: 0.7773399014778325
F1 Score: 0.7731504164625184
🏃 View run stylish-pig-529 at: https://dagshub.com/DhruvMishra007/mlops-mini-project.mlflow/#/experiments/1/runs/96d7b2e2d4ee4304b9d084a7213c5cc9
🧪 View experiment at: https://dagshub.com/DhruvMishra007/mlops-mini-project.mlflow/#/experiments/1
